In [2]:
import pandas as pd
import numpy as np
import os # Berguna untuk mengecek file gambar nanti

# Tentukan jumlah klien yang Anda inginkan (6-8)
N_CLIENTS = 8
# Seed untuk reproduktibilitas
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print(f"--- Menyiapkan Data untuk {N_CLIENTS} Klien ---")

# =============================================================================
# Langkah 1: Muat dan Filter Data
# =============================================================================
print("\n--- Langkah 1: Memuat dan Memfilter Data ---")

# Muat metadata
df = pd.read_csv('dataset/metadata.csv')
print(f"Total data awal: {len(df)} baris")

# Filter HANYA untuk data yang dikonfirmasi dengan biopsi
df_biopsed = df[df['biopsed'] == True].copy()
print(f"Data setelah filter 'biopsed == True': {len(df_biopsed)} baris")

# =============================================================================
# Langkah 2: Membuat Label Biner (Jinak vs. Ganas)
# =============================================================================
print("\n--- Langkah 2: Membuat Label Biner ---")

# Definisikan pemetaan label
# Berdasarkan kelas diagnostik:
# Ganas (Malignant) = 1: 'BCC' (Basal Cell Carcinoma), 'SCC' (Squamous Cell Carcinoma), 'MEL' (Melanoma)
# Jinak (Benign) = 0: 'NEV' (Nevus), 'SEK' (Seborrheic Keratosis)
#
# 'ACK' (Actinic Keratosis) adalah pra-kanker.
# Untuk PoC biner "jinak vs ganas", yang paling bersih adalah MENGECUALIKAN 'ACK'
# agar klasifikasinya tidak ambigu.

label_map = {
    'MEL': 1,
    'BCC': 1,
    'SCC': 1,
    'NEV': 0,
    'SEK': 0
}

df_biopsed['label'] = df_biopsed['diagnostic'].map(label_map)

# Buang baris yang labelnya NaN (yaitu, kelas 'ACK' yang kita kecualikan)
df_labeled = df_biopsed.dropna(subset=['label']).copy()

# Konversi label ke integer
df_labeled['label'] = df_labeled['label'].astype(int)

print(f"Data setelah membuang kelas ambigu ('ACK'): {len(df_labeled)} baris")
print("Distribusi kelas (Jinak vs. Ganas):")
print(df_labeled['label'].value_counts())

# =============================================================================
# Langkah 3: Partisi Data Federated (Split by Patient)
# =============================================================================
print(f"\n--- Langkah 3: Partisi Data ke {N_CLIENTS} Klien (berdasarkan Pasien) ---")

# 1. Dapatkan daftar pasien unik DARI data yang sudah dilabel
unique_patients = df_labeled['patient_id'].unique()
print(f"Total pasien unik untuk dibagi: {len(unique_patients)}")

# 2. Acak daftar pasien
np.random.shuffle(unique_patients)

# 3. Bagi daftar pasien menjadi N_CLIENTS grup
# np.array_split akan membagi secara (relatif) merata
client_patient_lists = np.array_split(unique_patients, N_CLIENTS)

# 4. Buat pemetaan dari patient_id ke client_id
patient_to_client_map = {}
for client_id, patient_list in enumerate(client_patient_lists):
    for patient_id in patient_list:
        patient_to_client_map[patient_id] = client_id

# 5. Tetapkan client_id ke setiap baris data (gambar)
df_labeled['client_id'] = df_labeled['patient_id'].map(patient_to_client_map)

print("\nDistribusi data (gambar) per klien:")
print(df_labeled['client_id'].value_counts().sort_index())

print("\nDistribusi pasien per klien:")
print(df_labeled.drop_duplicates('patient_id')['client_id'].value_counts().sort_index())


# =============================================================================
# Langkah 4: Buat dan Simpan Manifes Final
# =============================================================================
print("\n--- Langkah 4: Menyimpan Manifes Final ---")

# Manifes ini adalah "otak" dari dataset Anda.
# Hanya berisi kolom-kolom yang diperlukan untuk melatih model.
final_columns = ['img_id', 'label', 'client_id', 'patient_id']
manifest_df = df_labeled[final_columns]

# Simpan ke file CSV baru
output_filename = 'federated_manifest.csv'
manifest_df.to_csv(output_filename, index=False)

print(f"Manifes data berhasil disimpan ke: {output_filename}")
print("Contoh 5 baris pertama dari manifes:")
print(manifest_df.head())

print("\n--- Selesai ---")
print(f"File '{output_filename}' siap digunakan untuk simulasi Federated Learning Anda.")

--- Menyiapkan Data untuk 8 Klien ---

--- Langkah 1: Memuat dan Memfilter Data ---
Total data awal: 2298 baris
Data setelah filter 'biopsed == True': 1342 baris

--- Langkah 2: Membuat Label Biner ---
Data setelah membuang kelas ambigu ('ACK'): 1164 baris
Distribusi kelas (Jinak vs. Ganas):
label
1    1089
0      75
Name: count, dtype: int64

--- Langkah 3: Partisi Data ke 8 Klien (berdasarkan Pasien) ---
Total pasien unik untuk dibagi: 693

Distribusi data (gambar) per klien:
client_id
0    152
1    157
2    166
3    141
4    136
5    149
6    128
7    135
Name: count, dtype: int64

Distribusi pasien per klien:
client_id
0    87
1    87
2    87
3    87
4    87
5    86
6    86
7    86
Name: count, dtype: int64

--- Langkah 4: Menyimpan Manifes Final ---
Manifes data berhasil disimpan ke: federated_manifest.csv
Contoh 5 baris pertama dari manifes:
                  img_id  label  client_id patient_id
1     PAT_46_881_939.png      1          3     PAT_46
4   PAT_684_1302_588.png      1 